# Phantom segmentation playground

A hands-on view of the synthetic benchmark. Everything here runs the *same* code
the study runs (`pipelines.run_flood_fill_two_stage` on the frozen 360-cell
phantom) — this notebook only adds sliders, caching and pictures, via
`phantom_lab.py` sitting next to it.

One segmentation takes about 3 seconds, so you can keep changing parameters and
re-running as much as you like.

**Contents**

| section | what you get |
|---|---|
| 1–2 | the phantom itself: the angular field and the true cells |
| 3–5 | the published parameter set, its scores, and which cells it gets wrong |
| **6** | **the live tuner — sliders for both methods, this is the main one** |
| 7 | sweep one parameter, and two at once as a heatmap |
| 8 | is a parameter set stable across seeds? |
| 9 | tune the KAM baseline and compare the two arms |

**Kernel:** use `~/miniconda3/envs/main/bin/python` — plain `python`/`python3`
do not have `disell` installed.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

BENCHMARK = Path.cwd()          # this notebook lives in benchmark/
if not (BENCHMARK / "phantom_lab.py").exists():
    BENCHMARK = Path("/home/adam/Documents/Scripts/packages/disell/benchmark")
sys.path.insert(0, str(BENCHMARK))

import json

import numpy as np
import pandas as pd
from IPython.display import display
import phantom_lab as lab

print(lab.__file__)

## 2. The phantom

The benchmark is scored against one frozen volume: 360 Laguerre cells in a
24 x 160 x 160 grid at 1.0 x 0.4 x 0.4 um spacing, with a two-channel angular
field (chi, phi) in degrees. The labels are known exactly, which is the whole
point — this is the only evidence that can measure segmentation *accuracy*.

It loads from `oracle_results/cache/phantom.npz` and is never regenerated.

In [ ]:
ph = lab.load_phantom()
ph

In [ ]:
voxel = float(np.prod(ph.spacing_um_zyx))
volumes = np.bincount(ph.labels.ravel())[1:] * voxel

print(f"cells               {ph.n_cells}")
print(f"voxel volume        {voxel:.3f} um^3")
print(f"cell volume         median {np.median(volumes):6.1f} um^3   "
      f"range {volumes.min():.1f} - {volumes.max():.1f}")
print(f"equivalent diameter median {2 * (3 * np.median(volumes) / (4 * np.pi)) ** (1/3):.2f} um")
print(f"field channels      {ph.field.shape[-1]}  (chi, phi in degrees)")

Just the phantom, no segmentation — the field on the left, the true cells on the right.

In [ ]:
lab.show_slice(ph, z=12);

## 3. The published parameter set

`lab.BEST_PARAMS` is the count-first winner from `analysis/count_first_v1`:
the parameter set with the smallest absolute cell-count error, broken by
identity F1. On seed 0 it recovers exactly 360 cells.

In [ ]:
lab.BEST_PARAMS

In [ ]:
result = lab.segment(ph, **lab.BEST_PARAMS, seed=0)
result

### What the scores mean

- `n_cells_pred` / `cell_count_error` — how many cells came out. This is the
  first thing the study's selection policy looks at.
- `identity_f1` — fraction of cells matched one-to-one, where a match needs
  60 % overlap *in both directions*. The honest "did we find the same objects"
  number, and it is well below 1 even when the count is exact.
- `ari`, `vi_*_bits` — partition agreement. ARI counts voxel *pairs*, so it is
  dominated by the large cells; don't tune on it alone.
- `boundary_assd_um` — mean distance between true and predicted interfaces.
  Reported as uncertainty, never allowed to outrank count or identity.

In [ ]:
scores = lab.score(ph, result)
for name, value in scores.items():
    print(f"{name:34s} {value}")

For reference, `analysis/count_first_v1/count_first_report.json` records
ARI 0.885012, identity F1 0.833333, boundary ASSD 0.041242 um for this
configuration. The numbers above should match exactly.

## 4. Look at the slices

Top row is the algorithm running: the angular field, the KAM computed from it,
and the flood-fill markers before the watershed (grey is unassigned wall).
Bottom row is the comparison: truth, prediction, and both sets of boundaries
overlaid.

In the prediction panel each predicted cell is painted with the colour of the
true cell it overlaps most, so the two label panels can be compared by eye.
Two predicted cells sharing one colour means that true cell was **split**.

In the boundary panel: **yellow** = truth and prediction agree, **white** = a
true interface the segmentation missed, **red** = an interface it invented.

In [ ]:
lab.show_slice(ph, result, z=12);

In [ ]:
lab.show_slices(ph, result, z_values=[4, 12, 20]);

## 5. Which cells went wrong, and how

Every ground-truth cell coloured by its outcome:

- **recovered** — matched one-to-one at 60 % overlap both ways;
- **split** — two or more predicted cells each took a substantial share of it;
- **merged** — the predicted cell covering it also substantially covers another
  true cell;
- **missed** — neither, so nothing recognisable came out.

With an exactly correct cell *count* you can still have plenty of splits and
merges — they cancel in the count. That is why identity F1 is the tie-breaker.

In [ ]:
diagnosis = lab.cell_diagnosis(ph, result)
diagnosis["counts"]

In [ ]:
lab.show_errors(ph, result, z_values=[4, 12, 20], diagnosis=diagnosis);

Are the failures size-dependent? Small cells are what the minimum-size and threshold parameters fight over.

In [ ]:
frame = pd.DataFrame({
    "volume_um3": volumes,
    "status": [str(s) for s in diagnosis["status"][1:]],
})
frame.groupby("status")["volume_um3"].describe()[["count", "min", "50%", "max"]]

---
## 6. The live tuner

Sliders for both methods, with the picture and the scores beside them.

**How to drive it**

- The **flood fill / KAM threshold** toggle switches which method you are
  tuning. Each arm keeps its own sliders and its own values, so switching
  back and forth never loses your settings — worth knowing because
  `min_cell_size` means something quite different to the two arms (~116
  voxels for the flood fill, ~10 for KAM components).
- Nothing runs until you press **Run** (a segmentation takes a couple of
  seconds). Moving the **z** slider or changing the **view** only redraws what
  is already computed, which is instant.
- **Keep** stores the current run; **Clear kept** empties the list. Then
  `tuner.table()` scores everything you kept side by side.
- **Reset** puts the sliders back to the published values.
- The status line under the buttons is the live score. An invalid parameter set
  (no accepted markers) says so in red rather than raising.

**Where to start.** Change one slider at a time and watch the cell count.
`local_threshold_deg` and `min_cell_size` move the count the hardest;
`footprint_tolerance` mostly trades cells for cleaner walls; `kam_radius_um`
barely moves the count at all but does move the boundaries. Open
*what the parameters do* at the bottom for the full list.

In [ ]:
tuner = lab.tune(ph)
tuner

The tuner is a live object — anything the sliders do, you can also do in code,
and the two stay in sync.

In [ ]:
print("method: ", tuner.method)
print("params: ", tuner.params)
print("last run:", tuner.result)

Score everything you pressed **Keep** on:

In [ ]:
tuner.keep("whatever the sliders say now")   # or press the Keep button
tuner.table()

To carry a promising setting out of the tuner, just take `tuner.params` and
segment with it directly:

In [ ]:
mine = lab.run(ph, tuner.params, method=tuner.method, seed=0)
lab.score_table(ph, {"published": result, "mine": mine})[lab.KEY_METRICS]

---
## 7. Systematic sweeps

Sliders are good for getting a feel; sweeps are how you actually pin a value
down. `lab.sweep` varies one parameter and scores every value. Configurations
that produce no accepted markers are kept as a row of NaN with the reason —
they are genuinely invalid, not a bug.

In [ ]:
swept = lab.sweep(ph, "min_cell_size", [20, 60, 116, 180, 250])
lab.sweep_plot(swept);

In [ ]:
swept[lab.KEY_METRICS]

Note what that shows: identity F1 keeps *rising* past the point where the cell
count is right. Optimising F1 alone would talk you into 180+ and 26 missing
cells — which is exactly why the study's policy puts cell count first and uses
F1 only to break ties.

Log-spaced sweeps suit the threshold parameters:

In [ ]:
swept_local = lab.sweep(ph, "local_threshold_deg", np.round(np.logspace(-2.4, -1.7, 7), 5))
lab.sweep_plot(swept_local);

### Two parameters at once

`lab.grid` crosses two parameters and `lab.grid_plot` draws the heatmap. Cost is
one segmentation per cell (~3 s), so keep the lists short — the 3 x 3 below is
about half a minute. Blue is too few cells, red too many, white about right.

In [ ]:
crossed = lab.grid(ph, "min_cell_size", [60, 116, 180],
                       "footprint_tolerance", [0.04, 0.2, 0.4])
lab.grid_plot(crossed);

In [ ]:
lab.grid_plot(crossed, metric="identity_f1");

---
## 8. Is it stable across seeds?

`seed` only changes the order candidate seeds are visited in. A parameter set
whose cell count swings with the seed is not really a 360-cell solution — the
study screens on five seeds for exactly this reason. The published set hits the
count exactly on 3 of 5 seeds and is off by 2 on the other two.

Worth checking on anything you find in the tuner before believing it.

In [ ]:
seeds = {f"seed {s}": lab.segment(ph, **lab.BEST_PARAMS, seed=s) for s in range(5)}
lab.score_table(ph, seeds)[["n_cells_pred", "cell_count_error", "identity_f1", "ari"]]

---
## 9. The KAM baseline

The comparator: threshold the KAM field, take the connected components below
the threshold as markers, and run the *same* watershed. Same refinement step, so
the comparison is purely about how markers are found.

Two knobs matter — `percentile` (raise it to admit more voxels as cell interior)
and `min_cell_size`. Switch the tuner above to **KAM threshold** to feel them,
or cross them here.

`lab.KAM_BASELINE_PARAMS` is roughly the best a short hand search finds. It is a
starting point for tuning, **not** a published number: the paper's KAM
comparator is the cell-count-matched partition in
`continuation_results/kam_analysis/`.

In [ ]:
lab.KAM_BASELINE_PARAMS

In [ ]:
kam_grid = lab.grid(ph, "min_cell_size", [10, 30, 60],
                        "percentile", [35, 45, 55],
                    method="KAM threshold")
lab.grid_plot(kam_grid);

The best a short hand search finds for the KAM arm is 306 of 360 cells —
well short, and the grid above shows the count falling away in every
direction from there. A plausible reason is the phantom's incomplete
walls (`PhantomConfig.incomplete_wall_fraction` is 0.35): where a wall's
KAM ridge fades out, no closed low-KAM component forms, so two cells share
one marker. Worth testing rather than assuming — sweep it further if the
gap matters to what you want to claim.

Either way, tune the baseline on its own terms before quoting it: use its
own best settings, not the flood fill's.

In [ ]:
baseline = lab.run(ph, lab.KAM_BASELINE_PARAMS, method="KAM threshold")
print(baseline, baseline.params)

lab.score_table(ph, {"flood fill": result, "KAM threshold": baseline})[lab.KEY_METRICS]

In [ ]:
lab.show_slice(ph, baseline, z=12);

Cell-size distributions are a good check on whether a method is losing the small cells:

In [ ]:
lab.show_size_distribution(ph, {"flood fill": result, "KAM threshold": baseline});

---
## Notes

- The phantom is **synthetic**. It selects parameters and measures accuracy; it
  is not measured material. The real 6.2 % volume has no ground truth and can do
  neither. See `PROVENANCE.md`.
- `phantom_lab.py` is a thin wrapper — the algorithm lives in `pipelines.py`,
  the metrics in `bench_metrics.py` and `object_orientation_metrics.py`, and
  the full search in `two_stage_oracle.py`.
- `lab.PARAMETER_GUIDE` holds the slider ranges and the one-line description of
  each parameter. Widen a range there and the tuner follows.
- `lab.match_to_truth()` recolours a prediction for display only. It is not a
  matching metric; several predicted cells can take the same colour, which is
  what over-segmentation looks like.
- Fixed run settings (`MAX_SEED_ATTEMPTS`, `STAGNATION_TOLERANCE`,
  `WATERSHED_CONNECTIVITY`) are at the top of `phantom_lab.py` and match the
  study.

---
# Part II — the benchmark study

Everything above explores one segmentation at a time. This part covers the
machinery built for the flood-fill-versus-KAM benchmark: the objective the
search optimises, the merge step, the strain series, and how to read the
results.

## 10. The objective: recovered@90 and contamination

Cell *count* turned out to be the wrong thing to optimise. Two metrics replaced it.

**`recovered_at_90`** — a true cell counts as recovered only when some predicted
cell is at least 90 % pure **and** at least 90 % complete for it. Above 50 % that
pairing is automatically one-to-one (a predicted cell holding most of a true cell
leaves too little for any other), so it needs no assignment problem and is cheap
enough to score every configuration in a search.

**`contamination`** — the fraction of labelled voxels sitting in a cell other than
their own. This grades *how wrong* the failures are, which counting cannot:
a predicted cell straddling two true cells 50/50 and one overreaching by 5 % are
both "one fusion", but only the first is badly wrong.

The two failure modes are not equivalent, and contamination encodes that for free:

- **split** — a true cell divided across several predicted cells. The merge step
  folds them back, and splitting moves no voxel into the wrong cell, so it costs
  **zero** contamination.
- **fused** — two true cells sharing one predicted cell. Nothing downstream can
  separate them, and it costs contamination in proportion to the damage.

In [ ]:
result = lab.segment(ph, **{k: v for k, v in lab.CAPPED_START.items()
                            if not k.startswith("merge")}, seed=0)
scores = lab.score(ph, result)
for name in lab.KEY_METRICS:
    print(f"{name:26s} {scores[name]}")

## 11. The merge step

Instead of controlling the cell count with a large `min_cell_size` — which buys
the count by *deleting* genuinely small cells — segment with a small one and fold
each fragment into the neighbour it is orientationally closest to.

The rule is conservative: only cells below `merge_size_voxels` are candidates,
and a fragment merges only if its mean orientation is within `merge_threshold_deg`
of the neighbour's. **Two large cells are never merged into each other**, so the
step can remove fragments but cannot dissolve a real interface.

In [ ]:
over = dict(local_threshold_deg=0.011956, global_threshold_deg=-1.0,
            footprint_tolerance=0.20, footprint_radius_um=1.15,
            min_cell_size=10, kam_radius_um=1.2)

runs = {"no merge": lab.segment(ph, **over, seed=0)}
for size in (100, 350, 800):
    for threshold in (0.02, 0.10, 0.30):
        runs[f"merge {size}/{threshold}"] = lab.segment(
            ph, **over, seed=0, merge_size_voxels=size, merge_threshold_deg=threshold)

lab.score_table(ph, runs)[["n_cells_pred", "recovered_at_90", "contamination",
                           "fused_true_cells", "strict_split_true_cells", "ari"]]

Each run records what the merge actually did:

In [ ]:
merged = lab.segment(ph, **over, seed=0, merge_size_voxels=350,
                     merge_threshold_deg=0.10)
merged.merge_diagnostics

## 12. The strain series

Four phantoms built from the measured DFXM trend in
[Zelenika et al., *Sci Rep* **15**, 8655 (2025)](https://doi.org/10.1038/s41598-025-88262-3):
cell size falls and misorientation rises with strain, while the size distribution
keeps its log-normal shape.

Only the microstructure changes across the series — wall geometry, intracell
structure and noise are held fixed — so any change in the best parameters is
attributable to the microstructure and not to a changed rendering model.

**6.2 % is extrapolated** beyond the paper's measured 0.6–4.6 % range.

In [ ]:
import strain_phantoms

strain_phantoms.summary(strain_phantoms.DEFAULT_OUT)

In [ ]:
series = lab.strain_series()
for key, phantom in series.items():
    print(f"{key}: {phantom}")

A slice through the finest and coarsest microstructures, for scale:

In [ ]:
for key in ("2p4", "6p2"):
    if key in series:
        lab.show_slice(series[key], z=12);

Segmenting one of them. Note the parameters that suit the primary phantom
(8.05 µm cells) are not the ones that suit these (4.2–5.0 µm cells) — which is
the whole point of the strain study.

In [ ]:
if "4p6" in series:
    sp = series["4p6"]
    r = lab.segment(sp, **over, seed=0, merge_size_voxels=200, merge_threshold_deg=0.10)
    print(r)
    display(lab.score_table(sp, {"4.6 % strain": r})[lab.KEY_METRICS])
    lab.show_slice(sp, r, z=12);

## 13. Reading the search results

The search writes JSONL stores keyed by configuration hash. These cells read
whatever has finished; they are safe to run mid-search.

In [ ]:
import capped_search as cs
from pathlib import Path

store = Path("capped_results")
for name in ("markers", "kam", "merge", "baseline", "final"):
    path = store / f"{name}.jsonl"
    n = len(cs.read_rows(path)) if path.exists() else 0
    print(f"{name:9s} {n:>7,} rows")

In [ ]:
rows = [r for r in cs.read_rows(store / "markers.jsonl") if r.get("status") == "ok"]
if rows:
    frame = pd.DataFrame(rows)
    best = frame.iloc[sorted(range(len(frame)),
                             key=lambda i: cs.strict_recovery_key(frame.iloc[i].to_dict()))]
    display(best[["footprint_radius_um", "footprint_tolerance", "local_threshold_deg",
                  "min_cell_size", "n_cells_pred", "recovered_at_90",
                  "contamination", "fused_true_cells", "ari"]].head(10))
else:
    print("no marker rows yet")

The final report, once the chain has finished, is
`analysis/benchmark_report_v1/BENCHMARK.md`, with the per-arm figures beside it.
`capped_results/capped_report.json` records the winner under **each** policy —
the headline one plus the two earlier formulations — so the effect of the
selection rule on the answer is auditable rather than hidden.

In [ ]:
report = store / "capped_report.json"
if report.exists():
    payload = json.loads(report.read_text())
    for policy, winners in payload["winners"].items():
        print(f"=== {policy} ===")
        display(pd.DataFrame(list(winners.values()))[
            ["arm", "n_cells_pred", "recovered_at_90", "contamination",
             "fused_true_cells", "ari"]])
else:
    print("the search has not written its report yet")